# 1. Environment preparation

`DataLoader`:https://docs.pytorch.org/docs/main/utils/data.html#torch.utils.data.DataLoader

In [3]:
import torch
import torch.nn as nn
import numpy as np
import random
import urllib.request
import zipfile
import os

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, resnet34, resnet50, resnet101, resnet152
from pathlib import Path
from PIL import Image
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader

# 2. Prepare for the data set

## (1) Download the dataset

In [5]:
url = "https://zenodo.org/records/5226945/files/cats_dogs_light.zip?download=1"
zip_path = Path('/content/cats_dogs_light.zip')

# Download only if the file is not
if not zip_path.exists():
  urllib.request.urlretrieve(url, zip_path)

print("Dataset is downloaded")

Dataset is downloaded


## (2) Read the images and convert them into tensors  
python formal tech doc: https://docs.python.org/3/library/zipfile.html#zipfile.ZipFile.namelist


In [6]:
preprocess = ResNet18_Weights.IMAGENET1K_V1.transforms()    # Image Processing Pipeline

dataset = []

with zipfile.ZipFile(zip_path) as archive:

  for path in archive.namelist():

    if "/test/" not in path.lower():
      continue

    filename = Path(path).name.lower()

    if not filename.endswith((".jpg", ".jpeg", ".png")):
      continue

    if filename.startswith("cat."):
      label = 0
    elif filename.startswith("dog."):
      label = 1
    else:
      continue

    with archive.open(path) as file:
      image = Image.open(file).convert("RGB")

    image = preprocess(image)

    dataset.append((image, label))

print("Number of images:", len(dataset))
print("Shape of one image:", dataset[0][0].shape)



Number of images: 400
Shape of one image: torch.Size([3, 224, 224])


## (3) Use a `DataLoader` to get batches

In [8]:
loader = DataLoader(
    dataset,
    batch_size = 4,
    shuffle = False
)

for images, labels in loader:
  print("Image batch shape:", image.shape)
  print("Label batch shape:", labels.shape)
  print("Labels:", labels)
  break

Image batch shape: torch.Size([3, 224, 224])
Label batch shape: torch.Size([4])
Labels: tensor([1, 1, 1, 1])


# 3. Set up device and random seed

In [ ]:
SEED = 42
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Set up model

In [ ]:
model.eval()
features = []

with.torch.inference_model():

  x = model.conv1(images)
  x = model.bn1(x)
  x = model.relu(x)
  x = model.maxpoool(x)

  for stage in [model.layer1, model.layer2, model.layer3, model.layer4]:
    for block in stage:
      x = block(x)

      feature = x.mean(dim=(2, 3))

      features.append(feature.cpu())